# Rapid estimation of insect biodiversity using AI-assisted image analysis

**Reproduction notebook for Cabra-Leykin, Keasar & Keasar**

This notebook reproduces every analysis in the manuscript, starting from the raw
trap scans. It is organised to follow the paper:

| notebook section | manuscript |
|---|---|
| 1. Visual features from trap scans | 2.3.2, 3.1 (Figure 3) |
| 2. GPS-VDI and validation against expert taxonomy | 2.3.3, 2.5, 3.3 (Figure 4) |
| 3. Sensitivity tests | 2.3.4, 3.2 (Tables 2-4, Figure 8) |
| 4. Field application across 323 traps | 2.6, 3.4 (Figure 5) |

**What is reproduced here, and what is not.** The five visual features are
recomputed from the trap images by this notebook - they are *not* read from a
precalculated table. Insect detection (Faster R-CNN, manuscript 2.3.1) is outside
the scope of the paper's claims and is not re-run; its output, the bounding boxes,
is an input here. The ecological mixed models of section 2.6 are in R and live in
the `R/` directory; this notebook produces the table they consume.


## 0. Setup

Runs on Google Colab or locally. On Colab the data is downloaded from the Mendeley
deposit; locally, set `GPSVDI_DATA_ROOT` to your data folder.

Feature extraction takes roughly 40 minutes for the 8,200 annotated arthropods, so
it is checkpointed to `features_annotated.parquet` and skipped on later runs.


In [ ]:
import os
import re
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

IN_COLAB = "google.colab" in sys.modules

# A Colab runtime deletes everything under /content when the session ends, so an interrupted
# run loses both the downloaded data and every cached intermediate, and the next session
# repeats the work from scratch. Point DRIVE_CACHE at a folder in your own Google Drive to
# keep them between sessions. Left empty, nothing is mounted and no permission is requested.
DRIVE_CACHE = ""          # e.g. "/content/drive/MyDrive/gpsvdi"

if IN_COLAB:
    REPO_ROOT = Path("/content/gps-vdi")
    repo_url = "https://github.com/chenKeasar/gps-vdi.git"
    !pip -q install opencv-python-headless pyarrow
    !git clone -q {repo_url} {REPO_ROOT}
    sys.path.insert(0, str(REPO_ROOT))

    if DRIVE_CACHE:
        from google.colab import drive
        drive.mount("/content/drive")
        cache_root = Path(DRIVE_CACHE)
        cache_root.mkdir(parents=True, exist_ok=True)
    else:
        cache_root = Path("/content")

    DATA_ROOT = cache_root / "data"
    os.environ.setdefault("GPSVDI_WORK", str(cache_root / "work"))
    fetch = REPO_ROOT / "fetch_data.py"
    !python {fetch} --dest {DATA_ROOT}
else:
    REPO_ROOT = Path(os.environ.get("GPSVDI_REPO_ROOT", Path.cwd()))
    sys.path.insert(0, str(REPO_ROOT))
    # Defaults to the conventional location that fetch_data.py fills; override with the
    # environment variable to point at a data folder you keep somewhere else.
    DATA_ROOT = Path(os.environ.get("GPSVDI_DATA_ROOT", REPO_ROOT / "data"))

# Anchored to the repository, not to the directory the kernel happened to start in.
WORK = Path(os.environ.get("GPSVDI_WORK", REPO_ROOT / "gpsvdi_output"))
WORK.mkdir(exist_ok=True, parents=True)

if not (DATA_ROOT / "insects.csv").exists():
    raise FileNotFoundError(
        f"No deposit data under {DATA_ROOT}.\n"
        f'Download it with:  python fetch_data.py --dest "{DATA_ROOT}"\n'
        "or set GPSVDI_DATA_ROOT to a folder you have already populated.")

from gpsvdi import extract_trap_features, FEATURE_SET

print("data root :", DATA_ROOT)
print("work dir  :", WORK)
print("features  :", FEATURE_SET)

### 0.1 Analysis parameters

These are the published settings (manuscript 2.3.3): a 5-position grid per feature
dimension, Gaussian kernel parameter 40, averaged over repeated subsamples.


In [ ]:
N_BINS   = 5      # grid positions per feature dimension
LAMBDA   = 40     # Gaussian kernel parameter
SEED     = 0
TRAPS_DIR = DATA_ROOT / "40traps"

rng = np.random.RandomState(SEED)


## 1. Visual features from the trap scans

Each detected arthropod is described by five features (manuscript 2.3.2):

1. **`log_insect_size`** - projected area. The arthropod is separated from the trap
   background using HSV statistics estimated from the whole trap image, the mask is
   cleaned with blur/dilate/erode and small-contour removal, and the surviving pixel
   count is normalised to 1200 dpi before taking logarithms.
2. **`rb_ratio`, `rg_ratio`** - total red-to-blue and red-to-green intensity over the
   crop, background included.
3. **`svd_ratio01`, `svd_ratio12`** - ratios of successive singular values of the
   greyscale crop, capturing how concentrated the image's spatial pattern is.

### 1.1 Inputs

The expert annotation table and the trap images, both from the deposit. `insects.csv`
supplies the taxonomic identification, the bounding box of each arthropod, the trap
filename and the scan resolution. It carries **no feature columns**: every feature used
below is recomputed from the pixels.


In [ ]:
# Expert taxonomic annotation, one row per identified arthropod. The deposited table carries
# the bounding box, the trap filename and the scan resolution, so no second table is needed here.
insects = pd.read_csv(DATA_ROOT / "insects.csv")

# One specimen was given two identifications. Joining such a row against a second table
# multiplies it - one box_id became four - which moves the Table 2 correlations in the third
# decimal. Reduce to one row per box_id, and say what is being dropped.
repeated = insects[insects.duplicated("box_id", keep=False)]
if len(repeated):
    conflicting = repeated.groupby("box_id")[["order", "family"]].nunique().gt(1).any(axis=1)
    print(f"{len(repeated)} repeated annotation rows over {repeated['box_id'].nunique()} box_ids"
          f"; {int(conflicting.sum())} with conflicting identifications (first kept)")
boxes = insects.drop_duplicates("box_id").copy()
assert not boxes["box_id"].duplicated().any()

# pixels_per_inch decides log_insect_size. A non-numeric value makes that feature NaN for every
# row, which the estimator reports as "N insects with NaN values were removed" and then carries
# on having silently discarded the whole dataset. Stop here rather than let that happen.
boxes["pixels_per_inch"] = pd.to_numeric(boxes["pixels_per_inch"], errors="coerce")
if boxes["pixels_per_inch"].isna().any():
    raise ValueError("pixels_per_inch does not parse; log_insect_size cannot be computed")

boxes["trap_file"] = boxes["parent_image_file_name"]

print(f"{len(boxes):,} annotated arthropods on {boxes.trap_file.nunique()} traps")
print(f"{boxes['order'].nunique()} orders, {boxes['family'].nunique()} families")

### 1.2 Recompute the features from the images

This is the step that makes the analysis self-contained: every feature value used
from here on is derived from a trap scan in this notebook.


In [ ]:
FEATURES_FILE = WORK / "features_annotated.parquet"

if FEATURES_FILE.exists():
    features = pd.read_parquet(FEATURES_FILE)
    print(f"loaded cached features for {len(features):,} arthropods")
else:
    parts, t0 = [], time.time()
    groups = list(boxes.groupby("trap_file"))
    for k, (trap_file, sub) in enumerate(groups, 1):
        got = extract_trap_features(
            TRAPS_DIR / trap_file,
            [(r.box_id, r.x, r.y, r.w, r.h) for r in sub.itertuples()],
            sub["pixels_per_inch"].iloc[0])
        got["trap_file"] = trap_file
        parts.append(got)
        done = time.time() - t0
        print(f"[{k:>2}/{len(groups)}] {trap_file:<28} {len(sub):>4} boxes | "
              f"{done/60:5.1f} min elapsed, ~{done/k*(len(groups)-k)/60:5.1f} min left")
    features = pd.concat(parts)
    features.to_parquet(FEATURES_FILE)
    print(f"computed {len(features):,} in {(time.time()-t0)/60:.1f} min")

data = boxes.set_index("box_id").join(features.drop(columns="trap_file"), rsuffix="_new")


### 1.3 Verification against the STARdbi pipeline

The features computed above should match the values STARdbi stored for the same
arthropods. Segmentation is expected to be *exact*. The other features agree only to
the precision at which the deposited table records them - some rows carry the full
float, others about nine significant digits - and the singular-value ratios carry a
further tolerance because LAPACK implementations differ in their last digits.

The stored values live in the deposited detection table, which covers the field survey.
Thirty of the forty annotated traps are field traps and can be checked this way; the other
ten are from a June 2022 round that the survey does not include, so no stored values for
them exist. The check therefore runs on the arthropods of those thirty traps. It is a
reimplementation check, not a validation: the validation in section 2 uses all forty traps
and the expert taxonomy, and every feature it consumes is the one recomputed above.

In [ ]:
# STARdbi's stored values for the same boxes, taken from the deposited detection table.
# It carries rows with no box_id (detections never sorted to a taxon) and a couple of exactly
# duplicated records, so reduce it to one row per identified box before aligning. The dtype
# cast matters: a float/int mismatch would make reindex return all-NaN and silently report
# that nothing was compared.
stored = (pd.read_csv(DATA_ROOT / "field_traps.csv", low_memory=False)
            .dropna(subset=["box_id"])
            .drop_duplicates("box_id")
            .assign(box_id=lambda d: d["box_id"].astype(data.index.dtype))
            .set_index("box_id"))
rows = []
for feature in FEATURE_SET + ["insect_size[pixels]"]:
    old = stored[feature].reindex(data.index).astype(float)
    new = data[feature].astype(float)
    keep = old.notna() & new.notna()
    diff = (old[keep] - new[keep]).abs()
    rows.append({"feature": feature, "n": int(keep.sum()),
                 "max_abs_diff": diff.max(), "median_abs_diff": diff.median(),
                 "exact": int((diff == 0).sum())})
verification = pd.DataFrame(rows).set_index("feature")
_checked = data.index.isin(stored.index)
print(f"{_checked.sum():,} of {len(data):,} annotated arthropods have stored values, "
      f"on {data.loc[_checked, 'trap_file'].nunique()} of {data.trap_file.nunique()} traps")
assert (verification["n"] > 0).all(), "index misalignment: nothing was compared"
verification

### 1.4 The hyperspace of visual features (Figure 3)

Taxonomic families occupy distinct regions of the feature space. The panels below
show a two-dimensional projection; contours enclose 95, 75, 50, 25 and 10 % of the
kernel-density mass, and the cyan stars mark the four density peaks, repeated across
panels so the families' positions can be compared.


In [ ]:
FAM_X, FAM_Y = "rg_ratio", "svd_ratio01"
FAMILIES = ["Cicadellidae", "Phoridae", "Psychodidae", "Mymaridae"]
PEAKS_X, PEAKS_Y = [0.21, 0.34, 0.47, 0.38], [0.63, 0.24, 0.37, 0.64]

plot_df = data.copy()
for col in (FAM_X, FAM_Y):                       # scale to the unit square
    lo, hi = plot_df[col].min(), plot_df[col].max()
    plot_df[col] = (plot_df[col] - lo) / (hi - lo)

cmap = plt.colormaps["turbo"]
with plt.style.context("default"):
    fig, axes = plt.subplots(2, 2, figsize=(9, 9), sharex=True, sharey=True,
                             layout="constrained")
    for ax, family in zip(axes.flat, FAMILIES):
        fam = plot_df[plot_df["family"] == family]
        sns.kdeplot(data=fam, x=FAM_X, y=FAM_Y, fill=True, cmap=cmap,
                    levels=100, thresh=0.09, ax=ax)
        sns.kdeplot(data=fam, x=FAM_X, y=FAM_Y, fill=False,
                    levels=[0.1, 0.25, 0.50, 0.75, 0.95],
                    colors="white", linewidths=0.6, ax=ax)
        ax.scatter(PEAKS_X, PEAKS_Y, c="cyan", marker="*", s=100)
        ax.set_facecolor(cmap(0))
        ax.set(xlabel="", ylabel="", xlim=(0.04, 0.64), ylim=(0.08, 0.81))
        ax.set_title(f"{family} - {len(fam)} individuals", fontsize=13, pad=14)
        ax.grid(False)
    fig.patch.set_facecolor("white")
    fig.supxlabel("Red-to-green intensity ratio (scaled)", fontsize=13)
    fig.supylabel(r"SVD ratio $\log(\log(\sigma_0/\sigma_1))$ (scaled)", fontsize=13)
    fig.savefig(WORK / "fig3_hyperspace.png", dpi=300, bbox_inches="tight")
plt.show()


## 2. GPS-VDI and validation against expert taxonomy

The Grid Pseudo-Species Visual Diversity Index (manuscript 2.3.3) scales each feature
to [0, 1] using the range over all assemblages, lays a grid of `N_BINS` positions per
dimension over the unit hypercube, and assigns each individual to nearby vertices
through a Gaussian kernel of width controlled by `LAMBDA`. Those vertices act as
pseudo-species, so the Shannon index applies. Because the estimate is sensitive to
sample size, it is averaged over repeated subsamples of fixed size.

### 2.1 Per-trap diversity

The table written here holds, for every trap, the expert Shannon index at order and
family level alongside four visual-diversity estimates: the grid estimator and the
three kernel-density variants used for the comparison in Table 2.


In [ ]:
from BiodiversityUtils.BiodiversityParameters import (
    biodiversity_parameters_15062026_v3_annotated as current_parameters)
from BiodiversityUtils.biodiversity_estimation import biodiversity_estimation
from BiodiversityUtils.csv_directory_to_experiments_data import csv_directory_to_experiments_data
from BiodiversityUtils.Engineered_features_utils.get_grid_parameters import get_grid_parameters
from BiodiversityUtils.biodiversity_estimators.GridScaler import GridScaler
from BiodiversityUtils.biodiversity_estimators.get_biodiversity_estimators_15062026 import (
    get_biodiversity_estimators_15062026 as get_biodiversity_estimators)
from BiodiversityUtils.entropy_estimators import (
    KdeEntropyEstimator, KdeLooEntropyEstimator, CombinedKdeEntropyEstimator, GridEntropyEstimator)

# the analysis table: metadata and taxonomy, with OUR features
analysis = data.reset_index()
# Section 1.1 drops the precalculated features from the coordinate table, so the join above
# brings OUR recomputed features in unsuffixed. Take the "_new" column only if a collision
# did occur, so the analysis table carries the recomputed values either way.
for feature in FEATURE_SET:
    if feature + "_new" in analysis.columns:
        analysis[feature] = analysis[feature + "_new"]
assert set(FEATURE_SET) <= set(analysis.columns), \
    sorted(set(FEATURE_SET) - set(analysis.columns))

# analyse_unit() reads these three eagerly and records them beside each trap's result. None
# of them enters a computation: at TRAP level the grouping is by parent_image_id, so the
# labels only travel through to the output. Section 4 sets meta_group_name the same way.
analysis["experiment_name"] = "annotated_insects"
analysis["image_meta_group"] = 0
analysis["meta_group_name"] = 0

ANALYSIS_DIR = WORK / "csv_files"
ANALYSIS_DIR.mkdir(exist_ok=True, parents=True)
analysis.to_csv(ANALYSIS_DIR / "annotated_insects.csv", index=False)

parameters = current_parameters()
parameters.update({"DATA_ROOT": WORK, "Results_path": WORK,
                   "merged_csv_file_path": ANALYSIS_DIR / "annotated_insects.csv",
                   "ANALYSIS_LEVELS": ["TRAP"]})

bundle = csv_directory_to_experiments_data(parameters)
parameters["Experiments"] = bundle["experiments"]
grid_file = WORK / (bundle["experiments_string"] + "_grid_parameters.csv")
get_grid_parameters(parameters, bundle["df"]).to_csv(grid_file)

estimators = get_biodiversity_estimators(
    parameters, np.random.RandomState(SEED), bandwidths={"engineered": 0.6},
    entropy_estimators=[
        {"estimator": KdeEntropyEstimator(0.6), "name": "kde"},
        {"estimator": KdeLooEntropyEstimator(0.6), "name": "kde_loo"},
        {"estimator": CombinedKdeEntropyEstimator(0.6), "name": "Combined"},
        {"estimator": GridEntropyEstimator(LAMBDA, N_BINS, n_features=5), "name": "Grid"}],
    grid_scaler=GridScaler(grid_parameters=pd.read_csv(grid_file)))

for level in ["order", "family"]:
    biodiversity_estimation(parameters=parameters, estimators=estimators, taxonomic_level=level)


### 2.2 Concordance with expert taxonomy (manuscript 3.3)

Pearson correlations between expert Shannon diversity and visual diversity, unweighted
and weighted by the square root of trap abundance. The published values are r = 0.80 and
0.81 at order level, 0.67 and 0.74 at family level.


In [ ]:
from scipy import stats
from BiodiversityUtils.weighted_correlation import weighted_corr
from BiodiversityUtils.analysing_utils.get_weights_and_sizes import get_weights_and_sizes

ESTIMATORS = ["Grid_engineered", "kde_engineered", "kde_loo_engineered", "Combined_engineered"]
rows = []
per_trap = {}
for level in ["order", "family"]:
    d = pd.read_csv(WORK / f"biodiversity_per_trap_{level}_"
                           f"{parameters['Experiments'][0]}_{parameters['Timestamp']}_{SEED}.csv")
    per_trap[level] = d
    weights, _ = get_weights_and_sizes(d["n_boxes"].to_numpy())
    taxonomic = d[f"entropy({level})"].to_numpy()
    for name in ESTIMATORS:
        visual = d[name].to_numpy()
        r, p = stats.pearsonr(taxonomic, visual)
        rows.append({"level": level, "estimator": name, "n_traps": len(d),
                     "r_unweighted": round(float(r), 4),
                     "r_weighted": round(float(weighted_corr(taxonomic, visual, weights)), 4),
                     "p_value": float(p)})

correlations = pd.DataFrame(rows)
correlations.to_csv(WORK / "validation_correlations.csv", index=False)
correlations


### 2.3 Figure 4

Each circle is a trap, its area proportional to the number of arthropods caught. The
two circled traps are shown as order-composition histograms: the low-diversity trap is
dominated by a single order, the high-diversity one is not.


In [ ]:
import matplotlib as mpl

d = per_trap["order"]
ORDERS = ["Diptera", "Hemiptera", "Hymenoptera", "Coleoptera", "Thysanoptera",
          "Lepidoptera", "Neuroptera", "Araneae", "Trombidiform"]
colors = dict(zip(ORDERS, ["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#EECA3B",
                           "#72B7B2", "#FF9DA6", "#9D755D", "#BAB0AC"]))

n = d["n_boxes"].to_numpy()
weights, sizes = get_weights_and_sizes(n)
x, y = d["entropy(order)"].to_numpy(), d["Grid_engineered"].to_numpy()
r, _ = stats.pearsonr(x, y)
rw = weighted_corr(x, y, weights)
lo, hi = int(np.argmin(x)), int(np.argmax(x))

fig, ax = plt.subplots(figsize=(7.4, 5.0), layout="constrained")
ax.scatter(x, y, s=sizes, facecolor="#3B6EA5", edgecolor="white", linewidth=0.7, alpha=0.85, zorder=3)
slope, intercept = np.polyfit(x, y, 1)
grid_x = np.linspace(x.min(), x.max(), 50)
ax.plot(grid_x, intercept + slope * grid_x, color="#888888", lw=1.2, zorder=2)
for idx in (lo, hi):
    ax.scatter([x[idx]], [y[idx]], s=sizes[idx], facecolor="none",
               edgecolor="#C44E52", lw=1.6, zorder=4)
ax.set_xlabel("Shannon index of order-level taxonomic diversity")
ax.set_ylabel("GPS-VDI visual diversity")
ax.set_title("Visual diversity tracks expert taxonomic diversity across 40 traps")
ax.annotate(f"r = {r:.2f}\nweighted r = {rw:.2f}", xy=(0.03, 0.97),
            xycoords="axes fraction", va="top", ha="left", fontsize=8)
ax.margins(0.10)

shown = []
for idx, box in [(lo, [0.055, 0.60, 0.21, 0.26]), (hi, [0.70, 0.17, 0.21, 0.26])]:
    counts = {o: d.iloc[idx][o] for o in ORDERS if o in d.columns and d.iloc[idx][o] > 0}
    counts = dict(sorted(counts.items(), key=lambda kv: -kv[1]))
    shown += [o for o in counts if o not in shown]
    inset = ax.inset_axes(box)
    inset.bar(range(len(counts)), list(counts.values()),
              color=[colors[o] for o in counts], edgecolor="white", linewidth=0.4)
    inset.set_xticks([]); inset.tick_params(labelsize=5)
    for spine in ("top", "right"):
        inset.spines[spine].set_visible(False)
    inset.set_title(f"trap {int(d.iloc[idx]['trap_id'])}  (n={int(n[idx])})", fontsize=6, pad=2)

ax.legend(handles=[mpl.patches.Patch(facecolor=colors[o], label=o) for o in shown],
          frameon=False, fontsize=6, loc="lower right", bbox_to_anchor=(0.665, 0.02),
          title="insect order", title_fontsize=6)
fig.savefig(WORK / "fig4_visual_vs_taxonomic.png", dpi=300)
plt.show()


## 3. Sensitivity tests

Manuscript section 2.3.4 lists four checks. The claim under test is not that the exact
correlation is reproducible, but that it is *qualitatively stable* - a high correlation
should not depend on a lucky choice of estimator, parameters, or feature subset.

Each test caches its results, so a reviewer can re-run one without recomputing the others.
Costs differ sharply: the ablation and permutation tests take minutes per run, whereas the
grid search at high resolution is expensive (k = 7 means 16,807 grid vertices per sample).

### 3.1 Estimator comparison (Table 2)

Already computed in section 2: the grid estimator is compared against kernel-density,
leave-one-out kernel-density, and their combination, at both taxonomic levels.


In [ ]:
table2 = correlations[correlations.estimator.isin(
    ["Grid_engineered", "kde_engineered", "kde_loo_engineered"])].copy()
table2["estimator"] = table2["estimator"].map(
    {"Grid_engineered": "Grid", "kde_engineered": "KDE", "kde_loo_engineered": "KDE_loo"})
table2 = table2[["level", "estimator", "r_unweighted", "r_weighted"]]
table2.to_csv(WORK / "table2_estimator_comparison.csv", index=False)
table2


### 3.2 Feature ablation (Table 3)

Each feature in turn is replaced by its mean across all arthropods, which removes its
information while leaving the dimensionality of the space unchanged, and the analysis is
re-run. A feature that matters shows a drop in correlation.


In [ ]:
ABLATION_DIR = WORK / "feature_set_test"
ABLATION_DIR.mkdir(exist_ok=True)
ablation_estimators = get_biodiversity_estimators(
    {**parameters, "Results_path": ABLATION_DIR}, np.random.RandomState(SEED),
    bandwidths={"engineered": 0.6},
    entropy_estimators=[{"estimator": GridEntropyEstimator(LAMBDA, N_BINS, n_features=5),
                         "name": "Grid"}],
    grid_scaler=GridScaler(grid_parameters=pd.read_csv(grid_file)))

for feature in FEATURE_SET:
    target = ABLATION_DIR / (f"biodiversity_per_trap_order_{parameters['Experiments'][0]}_"
                             f"{parameters['Timestamp']}_{SEED}_{feature}.csv")
    if target.exists():
        continue
    flattened = pd.read_csv(parameters["merged_csv_file_path"])
    flattened[feature] = flattened[feature].to_numpy().mean()
    biodiversity_estimation(df=flattened, parameters={**parameters, "Results_path": ABLATION_DIR},
                            estimators=ablation_estimators, taxonomic_level="order", tag=feature)

full = per_trap["order"]
r_full, _ = stats.pearsonr(full["entropy(order)"], full["Grid_engineered"])
rows = [{"removed_feature": "(none - all five)", "r": round(float(r_full), 4)}]
for feature in FEATURE_SET:
    d = pd.read_csv(ABLATION_DIR / (f"biodiversity_per_trap_order_{parameters['Experiments'][0]}_"
                                    f"{parameters['Timestamp']}_{SEED}_{feature}.csv"))
    r, _ = stats.pearsonr(d["entropy(order)"], d["Grid_engineered"])
    rows.append({"removed_feature": feature, "r": round(float(r), 4)})
table3 = pd.DataFrame(rows)
table3.to_csv(WORK / "table3_feature_ablation.csv", index=False)
table3


### 3.3 Order-permuted traps (Table 4)

Synthetic assemblages are built by permuting which trap each taxonomic group's individuals
came from. This preserves the visual-cue/taxon association but destroys the real per-trap
composition, so each permutation is a different "trap" with a different Shannon index. If the
method works for the right reasons, the correlation should survive.

Each seed is a separate run of the full estimator, so this is the test to reduce first if
compute is limited. Set `PERMUTATION_SEEDS` to taste.


In [ ]:
from BiodiversityUtils.DataManipulator import TrapOrderRandomizer

# Kept deliberately small so the notebook runs in a reasonable time. The manuscript used a
# larger set; raising this only tightens the null distribution, it does not change the test.
PERMUTATION_SEEDS = range(1, 6)
PERM_DIR = WORK / "randomized_orders"
PERM_DIR.mkdir(exist_ok=True)

perm_estimators = get_biodiversity_estimators(
    {**parameters, "Results_path": PERM_DIR}, np.random.RandomState(SEED),
    bandwidths={"engineered": 0.6},
    entropy_estimators=[{"estimator": GridEntropyEstimator(LAMBDA, N_BINS, n_features=5),
                         "name": "Grid"}],
    grid_scaler=GridScaler(grid_parameters=pd.read_csv(grid_file)))

for s in PERMUTATION_SEEDS:
    target = PERM_DIR / (f"biodiversity_per_trap_order_{parameters['Experiments'][0]}_"
                         f"{parameters['Timestamp']}_{s}.csv")
    if target.exists():
        continue
    biodiversity_estimation(parameters={**parameters, "Results_path": PERM_DIR},
                            estimators=perm_estimators, taxonomic_level="order",
                            data_manipulator=TrapOrderRandomizer(seed=s), seed=s)

perm_r = []
for p in sorted(PERM_DIR.glob("biodiversity_per_trap_order_*.csv")):
    d = pd.read_csv(p)
    r, _ = stats.pearsonr(d["entropy(order)"], d["Grid_engineered"])
    perm_r.append(float(r))

table4 = pd.Series({"n_permutations": len(perm_r), "median": np.median(perm_r),
                    "mean": np.mean(perm_r), "std": np.std(perm_r, ddof=1),
                    "min": np.min(perm_r), "max": np.max(perm_r)}).round(3)
table4.to_csv(WORK / "table4_synthetic_traps.csv")
table4


### 3.4 Grid search over (k, lambda) — Figure 8

The two hyper-parameters are swept jointly, to show that the result does not rest on a fortunate
choice of either one.

**What this notebook reproduces as shipped.** `K_VALUES` below is set to `[3, 4]`, so the figure
produced here is the **bottom two rows of published Figure 8**, and the cell takes about 15
minutes. The published operating point — k = 5, lambda = 40 — lies outside that range. Two
consequences are expected and are not discrepancies: no cell is highlighted in the figure, and
the largest value shown (0.67 at k = 4, lambda = 20) is below the 0.80 reported in the paper.
The correlation rises with k, and the two rows shown are the cheapest, weakest end of the sweep.

**To extend it.** Results are cached per (k, lambda), so rows already computed are never redone
and only new ones cost anything. Set `K_VALUES = [3, 4, 5]` to bring the published setting back
into the figure, or `[3, 4, 5, 6, 7]` for the complete published sweep. Measured on a 16-core
laptop, at 12 lambda values per row:

| k | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|
| time for that row | 3 min | 12 min | 38 min | 94 min | 3.4 h |

Adding k = 5 alone costs about 38 minutes; the complete sweep takes roughly 6 hours. Cost grows
as k^5.


In [ ]:
from matplotlib.patches import Rectangle

# Cost grows as k**5, so the shipped sweep is the two cheapest rows of published Figure 8
# (~15 min). The published setting k = 5 is NOT in this range, so no cell is highlighted below
# and the maximum shown is lower than the paper's 0.80 - see the text above. Rows are cached,
# so extending costs only the new rows: +38 min for k = 5, +94 min for k = 6, +3.4 h for k = 7.
K_VALUES = [3, 4]
LAMBDA_VALUES = [2, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 160]
GRID_DIR = WORK / "grid_search"
GRID_DIR.mkdir(exist_ok=True)

for k in K_VALUES:
    for lam in LAMBDA_VALUES:
        tag = f"{k}_{lam}"
        target = GRID_DIR / (f"biodiversity_per_trap_order_{parameters['Experiments'][0]}_"
                             f"{parameters['Timestamp']}_{SEED}_{tag}.csv")
        if target.exists():
            continue
        sweep_params = {**parameters, "Results_path": GRID_DIR, "LAMBDA": lam}
        biodiversity_estimation(
            parameters=sweep_params, taxonomic_level="order", tag=tag,
            estimators=get_biodiversity_estimators(
                sweep_params, np.random.RandomState(SEED), bandwidths={"engineered": 0.6},
                entropy_estimators=[{"estimator": GridEntropyEstimator(lam, k, n_features=5),
                                     "name": "Grid"}],
                grid_scaler=GridScaler(grid_parameters=pd.read_csv(grid_file))))

rows = []
for p in GRID_DIR.glob("*.csv"):
    m = re.search(r"_(\d+)_(\d+)\.csv$", p.name)
    if not m:
        continue
    d = pd.read_csv(p)
    r, _ = stats.pearsonr(d["entropy(order)"], d["Grid_engineered"])
    rows.append({"k": int(m.group(1)), "lambda": int(m.group(2)), "r": float(r)})
sweep = pd.DataFrame(rows).sort_values(["k", "lambda"])
sweep.to_csv(WORK / "grid_search_results.csv", index=False)
heat = sweep.pivot(index="k", columns="lambda", values="r")

fig, ax = plt.subplots(figsize=(8.6, 3.4), layout="constrained")
im = ax.imshow(heat.values, aspect="auto", origin="lower", cmap="viridis")
ax.set_xticks(range(len(heat.columns)), [str(c) for c in heat.columns])
ax.set_yticks(range(len(heat.index)), [str(i) for i in heat.index])
ax.set_xlabel(r"kernel parameter $\lambda$")
ax.set_ylabel("grid resolution $k$")
_published_shown = N_BINS in list(heat.index) and LAMBDA in list(heat.columns)
_title = ("Correlation with taxonomic diversity across grid resolution $k$ "
          "and kernel parameter $\\lambda$")
if not _published_shown:
    _title += (f"\nPublished setting ($k$ = {N_BINS}, $\\lambda$ = {LAMBDA}) lies outside "
               "the range swept here")
ax.set_title(_title, fontsize=10)
ax.grid(False)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.values[i, j]
        if pd.notna(v):
            c = im.cmap(im.norm(v))
            lum = 0.299 * c[0] + 0.587 * c[1] + 0.114 * c[2]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5,
                    color="white" if lum < 0.45 else "black")
if _published_shown:
    ax.add_patch(Rectangle((list(heat.columns).index(LAMBDA) - 0.5,
                            list(heat.index).index(N_BINS) - 0.5), 1, 1,
                           fill=False, edgecolor="red", linewidth=1.8))
fig.colorbar(im, ax=ax, label="Pearson r (order level)")
fig.savefig(WORK / "fig8_grid_search_heatmap.png", dpi=300)
plt.show()


## 4. Field application across the full trap set

Sections 1-3 used the 40 expert-annotated traps, where visual diversity can be checked against
taxonomy. This section applies the validated estimator to every trap in the survey, where no
taxonomy exists, and asks the three ecological questions of manuscript section 2.6.

Only the visual index is computed here. The mixed models behind Figures 6 and 7 are in R and
live in `R/`; this section ends by writing the table they consume.

### 4.1 Per-trap visual diversity

Features come from the deposited detection table `field_traps.csv`: the trap scans for the full
survey are too numerous to re-segment in a notebook run, and section 1.3 established that the
stored features and the recomputed ones agree to floating-point precision.


In [ ]:
FIELD_OUT = WORK / "field"
FIELD_OUT.mkdir(exist_ok=True, parents=True)

# The deposited detection table: one row per detected arthropod across the full survey,
# carrying the five features as computed by the production pipeline.
field = pd.read_csv(DATA_ROOT / "field_traps.csv", low_memory=False)
before = len(field)
field = field.dropna(subset=FEATURE_SET)
field["meta_group_name"] = 0
print(f"{before:,} detections -> {len(field):,} with complete features, "
      f"{field.parent_image_id.nunique()} traps")

field_csv = FIELD_OUT / "field_detections.csv"
field.to_csv(field_csv, index=False)

field_params = {**parameters, "DATA_ROOT": FIELD_OUT, "Results_path": FIELD_OUT,
                "merged_csv_file_path": field_csv, "Experiments": ["field"]}
field_grid = FIELD_OUT / "field_grid_parameters.csv"
get_grid_parameters(field_params, field).to_csv(field_grid)

field_estimator = get_biodiversity_estimators(
    field_params, np.random.RandomState(SEED), bandwidths={"engineered": 0.6},
    entropy_estimators=[{"estimator": GridEntropyEstimator(LAMBDA, N_BINS, n_features=5),
                         "name": "Grid"}],
    grid_scaler=GridScaler(grid_parameters=pd.read_csv(field_grid)))[0]

PER_TRAP = FIELD_OUT / "per_trap_raw.csv"
if PER_TRAP.exists():
    per_trap_field = pd.read_csv(PER_TRAP)
else:
    records, t0 = [], time.time()
    groups = list(field.groupby("parent_image_id"))
    for i, (trap_id, g) in enumerate(groups, 1):
        records.append({"trap_id": trap_id, "trap_name": g.iloc[0]["parent_image_file_name"],
                        "experiment": g.iloc[0]["experiment_name"],
                        "location": g.iloc[0]["location"],
                        "placement_date": g.iloc[0]["placement_date"],
                        "n_insects": len(g),
                        "visual_diversity": float(field_estimator.estimate_entropy(g).entropy)})
        if i % 50 == 0:
            print(f"  {i}/{len(groups)} traps | {(time.time()-t0)/60:.1f} min")
    per_trap_field = pd.DataFrame(records)
    per_trap_field.to_csv(PER_TRAP, index=False)
print(f"{len(per_trap_field)} traps")


### 4.2 Attach site, plot, habitat and season

Habitat is not recorded in the detection export. It is recovered from the trap filename, which
encodes `site_plot_replicate`, combined with `plot_metadata.csv` (manuscript Table 1).

Two details matter. Sites are matched to Table 1 **by name, not by position** - the orderings
differ, and matching by position silently mislabels several sites. And within a site, the
natural plot is always plot 3: exactly those sites with three plots have one, which is the
"11 sites sampled in both habitats" of section 2.1.


In [ ]:
import difflib

# manuscript Table 1; deposited with the data when available, otherwise the repo copy
_pm = DATA_ROOT / "plot_metadata.csv"
if not _pm.exists():
    _pm = REPO_ROOT / "plot_metadata.csv"
plot_meta = pd.read_csv(_pm)
plots_per_site = plot_meta.groupby("site").size().to_dict()

pattern = re.compile(r"^(\d+)_(\d+)_(\d+)_")
parsed = per_trap_field.trap_name.map(
    lambda s: pattern.match(s) if isinstance(s, str) else None)
per_trap_field["site_n"] = [int(m.group(1)) if m else np.nan for m in parsed]
per_trap_field["plot_n"] = [int(m.group(2)) if m else np.nan for m in parsed]
per_trap_field["rep_n"] = [int(m.group(3)) if m else np.nan for m in parsed]

def normalise(name):
    s = str(name).lower().replace("(", " ").replace(")", " ")
    return re.sub(r"\s+", " ", re.sub(r"[^a-z ]", " ", s)).strip()

lookup = {normalise(s): s for s in plot_meta.site.unique()}
site_of = {}
for site_n, loc in per_trap_field.groupby("site_n")["location"].agg(
        lambda s: s.mode().iat[0]).items():
    key = normalise(loc)
    if key in lookup:
        site_of[site_n] = lookup[key]
    else:
        close = difflib.get_close_matches(key, list(lookup), n=1, cutoff=0.7)
        site_of[site_n] = lookup[close[0]] if close else None

per_trap_field["site"] = per_trap_field.site_n.map(site_of)
per_trap_field["n_plots_in_site"] = per_trap_field.site.map(plots_per_site)
per_trap_field["habitat"] = np.where(
    (per_trap_field.plot_n == 3) & (per_trap_field.n_plots_in_site == 3),
    "Natural", "Agriculture")
per_trap_field.loc[per_trap_field.site.isna(), "habitat"] = np.nan
# The deposited dates are day-first. Without an explicit format pandas infers month-first
# from the early rows and then either raises or, where both fields are <= 12, silently
# swaps them - which would move traps between spring and summer.
month = pd.to_datetime(per_trap_field.placement_date, format="%d/%m/%Y").dt.month
per_trap_field["season"] = np.where(month.isin([4, 5, 6]), "spring", "summer")
per_trap_field["diversity_z"] = (
    (per_trap_field.visual_diversity - per_trap_field.visual_diversity.mean())
    / per_trap_field.visual_diversity.std(ddof=1))

unplaced = per_trap_field.site.isna().sum()
if unplaced:
    print(f"WARNING: {unplaced} trap(s) could not be placed - non-standard filename:")
    print(per_trap_field[per_trap_field.site.isna()].trap_name.tolist())
per_trap_field.groupby(["habitat", "season"]).size()


### 4.3 Abundance versus diversity (Figure 5)

The first ecological question: does catching more insects mean catching a more diverse sample?


In [ ]:
placed = per_trap_field.dropna(subset=["site"])
r, p = stats.pearsonr(placed.n_insects, placed.visual_diversity)
n = len(placed)
zf = np.arctanh(r); se = 1 / np.sqrt(n - 3)
ci_lo, ci_hi = np.tanh(zf - 1.96 * se), np.tanh(zf + 1.96 * se)
print(f"abundance vs diversity: r = {r:.3f}, p = {p:.2e}, n = {n}, 95% CI [{ci_lo:.3f}, {ci_hi:.3f}]")

triplets = placed.groupby(["site", "plot_n", "experiment"]).agg(
    n_insects=("n_insects", "sum"), visual_diversity=("visual_diversity", "mean")).reset_index()
r_trip, p_trip = stats.pearsonr(triplets.n_insects, triplets.visual_diversity)
print(f"triplicate-pooled     : r = {r_trip:.3f}, p = {p_trip:.2e}, n = {len(triplets)}")

fig, ax = plt.subplots(figsize=(6.6, 4.4), layout="constrained")
colours = {"Agriculture": "#3B6EA5", "Natural": "#4C9A52"}
for habitat, g in placed.groupby("habitat"):
    ax.scatter(g.n_insects, g.visual_diversity, s=26, alpha=0.8, linewidth=0.5,
               facecolor=colours[habitat], edgecolor="white", label=habitat, zorder=3)
slope, intercept = np.polyfit(placed.n_insects, placed.visual_diversity, 1)
grid_x = np.linspace(placed.n_insects.min(), placed.n_insects.max(), 60)
ax.plot(grid_x, intercept + slope * grid_x, color="#888888", lw=1.3, zorder=2)
ax.set_xlabel("Insects caught per trap")
ax.set_ylabel("GPS-VDI visual diversity")
ax.set_title("Richly populated traps are consistently less diverse")
ax.annotate(f"r = {r:.2f}   n = {n}\n95% CI [{ci_lo:.2f}, {ci_hi:.2f}]", xy=(0.97, 0.97),
            xycoords="axes fraction", ha="right", va="top", fontsize=7.5)
ax.legend(frameon=False, loc="lower left", fontsize=7.5, title="habitat", title_fontsize=7.5)
ax.margins(0.06)
fig.savefig(WORK / "fig5_abundance_vs_diversity.png", dpi=300)
plt.show()


### 4.4 Hand-off to the R analysis

Everything the mixed models need is written to one file. **This is the only interface between
this notebook and the R code** - nothing here imports or calls R, and deleting `R/` leaves
sections 1-4 fully runnable.

| column | meaning |
|---|---|
| `trap_id` | STARdbi identifier for the trap image |
| `trap_name` | scan filename, `site_plot_replicate_dates.jpg` |
| `experiment` | sampling round as recorded in STARdbi |
| `site` | site name, matched to manuscript Table 1 |
| `plot_n`, `rep_n` | plot within site, replicate within plot |
| `habitat` | `Natural` or `Agriculture` |
| `season` | `spring` (Apr-Jun) or `summer` (Jul-Sep) |
| `n_insects` | arthropods detected on the trap (abundance) |
| `visual_diversity` | GPS-VDI index |
| `diversity_z` | z-transformed diversity, as used by the models |


In [ ]:
HANDOFF = WORK / "per_trap_diversity.csv"
per_trap_field.to_csv(HANDOFF, index=False)
print(f"wrote {HANDOFF} - {len(per_trap_field)} rows, {len(per_trap_field.columns)} columns")
print("\nThe R scripts in R/ read this file; see R/README_R.md.")
per_trap_field.head()
